In [1]:
import mysql.connector
my=mysql.connector.connect(
    user='root',
    host='localhost',
    password=''
)
mycursor=my.cursor(buffered=True)
my.commit()

In [11]:
mycursor.execute("show databases")

In [5]:
for i in mycursor:
    print(i)

('global_electronic',)
('information_schema',)
('mysql',)
('performance_schema',)
('phpmyadmin',)
('project',)
('test',)


In [2]:
mycursor.execute('use Global_electronic')
my.commit()

### Gender distribution -------------

In [27]:
mycursor.execute('select Gender,count(*) from Global_electronic.customer group by Gender')
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,[i[0] for i in mycursor.description],tablefmt='psql'))

+----------+------------+
| Gender   |   count(*) |
|----------+------------|
| Female   |       7518 |
| Male     |       7748 |
+----------+------------+


# gender, age, location distribution ----------

In [5]:
mycursor.execute('''select 
                 continent,country,state,
                 case 
                 when age <18 then "under 18"
                 when age between 18 and 25 then "age 18 to 25"
                 when age between 26 and 35 then "age 26 to 35"
                 when age between 36 and 45 then "age 36 to 45"
                 when age between 46 and 60 then "age 46 to 60"
                 when age between 61 and 75 then "age 61 to 75"
                 else "above 75"
                 end as age_group,gender,
                 count(*) as no_of_customer from Global_electronic.customer group by age_group,continent,country,state,gender
                 order by age_group,continent,country,state,gender
                 
                 ''')
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,[i[0] for i in mycursor.description],tablefmt='psql'))

+---------------+----------------+------------------------------+--------------+----------+------------------+
| continent     | country        | state                        | age_group    | gender   |   no_of_customer |
|---------------+----------------+------------------------------+--------------+----------+------------------|
| Australia     | Australia      | Australian Capital Territory | above 75     | Female   |                2 |
| Australia     | Australia      | Australian Capital Territory | above 75     | Male     |                1 |
| Australia     | Australia      | New South Wales              | above 75     | Female   |               43 |
| Australia     | Australia      | New South Wales              | above 75     | Male     |               48 |
| Australia     | Australia      | Northern Territory           | above 75     | Male     |                2 |
| Australia     | Australia      | Queensland                   | above 75     | Female   |               22 |
|

### average order value ---------------

In [21]:
mycursor.execute('''select sales.productkey,sales.quantity,product.unit_price_usd,
                 avg(sales.quantity * product.unit_price_usd) as average_order_value
                 from sales inner join product on sales.productkey = product.productkey
                 GROUP BY sales.productkey, sales.quantity, product.unit_price_usd''')
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,[i[0] for i in mycursor.description],tablefmt='psql'))


+--------------+------------+------------------+-----------------------+
|   productkey |   quantity |   unit_price_usd |   average_order_value |
|--------------+------------+------------------+-----------------------|
|            1 |          1 |            12.99 |                 12.99 |
|            1 |          2 |            12.99 |                 25.98 |
|            1 |          3 |            12.99 |                 38.97 |
|            1 |          4 |            12.99 |                 51.96 |
|            1 |          5 |            12.99 |                 64.95 |
|            1 |          7 |            12.99 |                 90.93 |
|            1 |          8 |            12.99 |                103.92 |
|            1 |         10 |            12.99 |                129.9  |
|            2 |          1 |            12.99 |                 12.99 |
|            2 |          2 |            12.99 |                 25.98 |
|            2 |          3 |            12.99 |   

### Frequency order ----------------------

In [23]:
mycursor.execute('''select customer.customerkey,count(sales.order_number) as order_count
                  from customer inner join sales on customer.customerkey = sales.customerkey 
                 group by customer.customerkey''')
out=mycursor.fetchall()
from tabulate import tabulate
print(tabulate(out,[i[0] for i in mycursor.description],tablefmt='psql'))

+---------------+---------------+
|   customerkey |   order_count |
|---------------+---------------|
|           301 |             1 |
|           325 |            10 |
|           554 |             4 |
|          1042 |             3 |
|          1314 |             5 |
|          1568 |             9 |
|          1585 |             4 |
|          1626 |             4 |
|          1642 |             1 |
|          1863 |             8 |
|          2238 |             3 |
|          2248 |             2 |
|          2435 |             4 |
|          2469 |             2 |
|          2512 |             5 |
|          2792 |             1 |
|          3002 |             3 |
|          3203 |             8 |
|          3327 |             3 |
|          3575 |             1 |
|          3964 |             1 |
|          4174 |             5 |
|          5097 |             1 |
|          5445 |             1 |
|          5962 |             8 |
|          6258 |             3 |
|          630

### Most preferred products ---------------

In [39]:
mycursor.execute('''
                 select product.product_name,sales.productkey,count(sales.productkey) as preferred_product 
                 from sales 
                 inner join product on sales.productkey = product.productkey
                 group by sales.productkey,product.productkey 
                 order by preferred_product desc
                    ''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------------------------------------------------------------------------+--------------+---------------------+
| product_name                                                                        |   productkey |   preferred_product |
|-------------------------------------------------------------------------------------+--------------+---------------------|
| Adventure Works Desktop PC2.30 MD230 Black                                          |          423 |                 162 |
| WWI Desktop PC1.80 E1800 White                                                      |          458 |                 158 |
| WWI Desktop PC1.60 E1600 Black                                                      |          446 |                 158 |
| Adventure Works Desktop PC2.30 MD230 White                                          |          434 |                 158 |
| WWI Desktop PC1.80 E1801 Black                                                      |          448 |                 157 |


### Age group by customer purchasing behaviour -------------

In [7]:
mycursor.execute('''
                 select 
                 case 
                 when age <18 then"under 18"
                 when age between 19 and 25 then "age 19 to 25"
                 when age between 26 and 35 then "age 26 to 35"
                 when age between 36 and 45 then "age 36 to 45"
                 when age between 46 and 60 then "age 46 to 60"
                 else "above 60"
                 end as age_group,sales.productkey from customer
                 inner join sales on customer.customerkey = sales.customerkey
                 group by age_group,sales.customerkey,sales.productkey 
                 order by age_group,sales.customerkey,sales.productkey desc''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+--------------+
| age_group    |   productkey |
|--------------+--------------|
| above 60     |           53 |
| above 60     |         1634 |
| above 60     |         1284 |
| above 60     |          166 |
| above 60     |           24 |
| above 60     |         1310 |
| above 60     |           67 |
| above 60     |         2132 |
| above 60     |         1236 |
| above 60     |          202 |
| above 60     |         1419 |
| above 60     |          547 |
| above 60     |            5 |
| above 60     |         1424 |
| above 60     |         2137 |
| above 60     |         1914 |
| above 60     |         1178 |
| above 60     |          741 |
| above 60     |          429 |
| above 60     |          267 |
| above 60     |          130 |
| above 60     |          116 |
| above 60     |         1649 |
| above 60     |          709 |
| above 60     |          459 |
| above 60     |           65 |
| above 60     |         1648 |
| above 60     |         1519 |
| above 

### highest sales on country ----------

In [8]:
mycursor.execute('''
                 select country,count(sales.order_number) as no_of_sales  from customer
                 inner join sales on customer.customerkey = sales.customerkey 
                 group by  country 
                 order by no_of_sales desc              
''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+----------------+---------------+
| country        |   no_of_sales |
|----------------+---------------|
| United States  |         33767 |
| United Kingdom |          8140 |
| Germany        |          5956 |
| Canada         |          5415 |
| Australia      |          2941 |
| Italy          |          2685 |
| Netherlands    |          2250 |
| France         |          1730 |
+----------------+---------------+


### profit on product wise ------------

In [14]:
mycursor.execute('''
                select product.productkey,sum(sales.quantity) as total_quantity,
                 sum((unit_price_usd - unit_cost_usd) * sales.quantity) as profit
                 from product 
                 left join sales on product.productkey = sales.productkey
                 group by productkey
                 order by profit desc                
''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+------------------+-----------+
|   productkey |   total_quantity |    profit |
|--------------+------------------+-----------|
|          444 |              550 | 337986    |
|          416 |              481 | 311664    |
|          428 |              479 | 310368    |
|          422 |              462 | 299353    |
|          433 |              451 | 292225    |
|          455 |              462 | 283908    |
|          450 |              460 | 282679    |
|          438 |              392 | 240892    |
|          434 |              521 | 168564    |
|          423 |              514 | 166300    |
|          417 |              480 | 155299    |
|          439 |              458 | 138289    |
|          451 |              456 | 137685    |
|          456 |              448 | 135269    |
|          429 |              405 | 131232    |
|          427 |              476 | 120580    |
|          421 |              464 | 117540    |
|          420 |              477 | 1168

### Total sales on categories and subcategories ----------

In [4]:
mycursor.execute('''
                select product.category,product.subcategory,sum(sales.quantity * product.unit_price_usd) as total_sales
                 from product 
                 inner join sales on product.productkey  = sales.productkey
                 group by product.category,product.subcategory 
                 order by total_sales desc            
''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------------------+----------------------------------+------------------+
| category                      | subcategory                      |      total_sales |
|-------------------------------+----------------------------------+------------------|
| Computers                     | Desktops                         |      9.90636e+06 |
| Home Appliances               | Water Heaters                    |      3.15451e+06 |
| Music, Movies and Audio Books | Movie DVD                        |      3.13101e+06 |
| Cell phones                   | Touch Screen Phones              |      3.08346e+06 |
| TV and Video                  | Televisions                      |      2.99602e+06 |
| Cameras and camcorders        | Camcorders                       |      2.9914e+06  |
| Computers                     | Laptops                          |      2.963e+06   |
| Cell phones                   | Smart phones & PDAs              |      2.80566e+06 |
| Computers                     

### Month wise revenue ---------------

In [10]:
mycursor.execute('''
                select  date_format(order_date,'%y-%m') as month,
                 round(sum(sales.quantity * product.unit_price_usd),2) as total_revenue
                 from sales
                 inner join product on sales.productkey = product.productkey
                 group by date_format(order_date,'%y-%m') 
                 order by date_format(order_date,'%y-%m')

''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+---------+------------------+
| month   |    total_revenue |
|---------+------------------|
| 16-01   | 552027           |
| 16-02   | 743305           |
| 16-03   | 272866           |
| 16-04   | 101469           |
| 16-05   | 490377           |
| 16-06   | 474365           |
| 16-07   | 406173           |
| 16-08   | 462298           |
| 16-09   | 498106           |
| 16-10   | 546173           |
| 16-11   | 554938           |
| 16-12   | 884168           |
| 17-01   | 604864           |
| 17-02   | 683899           |
| 17-03   | 292945           |
| 17-04   |  41967.4         |
| 17-05   | 561930           |
| 17-06   | 534708           |
| 17-07   | 457241           |
| 17-08   | 527872           |
| 17-09   | 614100           |
| 17-10   | 584271           |
| 17-11   | 612697           |
| 17-12   |      1.13503e+06 |
| 18-01   | 818048           |
| 18-02   |      1.14114e+06 |
| 18-03   | 359670           |
| 18-04   |  65705.4         |
| 18-05   | 958555           |
| 18-06 

### Highest revenue generated by stores ------------

In [3]:
mycursor.execute('''
                select storekey,round(sum(sales.quantity * product.unit_price_usd),2) as total_revenue                 
                 from sales
                 inner join product on sales.productkey = product.productkey
                 group by storekey
                 order by total_revenue desc
''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+------------+------------------+
|   storekey |    total_revenue |
|------------+------------------|
|          0 |      1.02733e+07 |
|         55 |      1.24953e+06 |
|         54 |      1.24549e+06 |
|         50 |      1.24208e+06 |
|         45 |      1.22318e+06 |
|          9 |      1.21104e+06 |
|         59 |      1.15956e+06 |
|         61 |      1.15867e+06 |
|         57 |      1.15555e+06 |
|         65 |      1.1361e+06  |
|         64 |      1.11808e+06 |
|          8 |      1.10384e+06 |
|         43 |      1.09764e+06 |
|         66 |      1.08966e+06 |
|         44 |      1.08266e+06 |
|         56 |      1.07327e+06 |
|         51 |      1.05003e+06 |
|         53 |      1.04946e+06 |
|         47 |      1.03309e+06 |
|         48 | 959139           |
|         10 | 949411           |
|         38 | 877537           |
|         49 | 854364           |
|         40 | 839274           |
|         30 | 836825           |
|         29 | 825296           |
|         42 |

## Currency impact on sales based on exchange rate ---------

In [29]:
mycursor.execute('''
                select currency_code,sum(sales.quantity * exchange.exchange) as sales_bsed_on_currency
                 from sales
                 inner join exchange on sales.currency_code = exchange.`currency code`
                 and date(sales.order_date) = date(exchange.date)       
                 group by sales.currency_code         
''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------+--------------------------+
| currency_code   |   sales_bsed_on_currency |
|-----------------+--------------------------|
| AUD             |                  12843.7 |
| CAD             |                  22119.9 |
| EUR             |                  35376.9 |
| GBP             |                  19464.7 |
| USD             |                 106407   |
+-----------------+--------------------------+


## Latest popular product ----------------

In [37]:
mycursor.execute('''
                select product.product_name,product.productkey,sum(sales.quantity) as total_quantity
                 from product 
                 inner join sales on sales.productkey = product.productkey
                 group by product.product_name,product.productkey
                 order by total_quantity desc
                 limit 10               
''')
out = mycursor.fetchall()

from tabulate import tabulate
print(tabulate(out, [i[0] for i in mycursor.description], tablefmt='psql'))

+--------------------------------------------+--------------+------------------+
| product_name                               |   productkey |   total_quantity |
|--------------------------------------------+--------------+------------------|
| WWI Desktop PC2.33 X2330 Black             |          444 |              550 |
| WWI Desktop PC1.80 E1800 White             |          458 |              538 |
| Adventure Works Desktop PC1.60 ED160 Black |          424 |              521 |
| Adventure Works Desktop PC2.30 MD230 White |          434 |              521 |
| Adventure Works Desktop PC1.80 ED180 Black |          425 |              520 |
| Adventure Works Desktop PC2.30 MD230 Black |          423 |              514 |
| WWI Desktop PC1.60 E1600 Black             |          446 |              509 |
| WWI Desktop PC1.60 E1600 Silver            |          440 |              507 |
| WWI Desktop PC1.80 E1801 Black             |          448 |              505 |
| WWI Desktop PC1.60 E1600 R